# 05. Monte Carlo Fixture Simulation

**Stage:** 05_simulation  
**Inputs:** `data/processed/adjusted_lambdas.parquet`, `data/processed/scoreline_probabilities.parquet`, `src/config/config.yaml`  
**Outputs:** `data/processed/mc_probabilities.parquet`  

This notebook executes stochastic Monte Carlo sampling (10,000 trials per fixture) from expected goal Poisson distributions, validates simulation convergence against exact analytical scoreline matrix probabilities, and saves output probabilities in 1-row-per-fixture layout.

## 1. Environment & Setup

In [ ]:
# Load configuration and imports
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Locate repo root dynamically
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "config" / "loader.py").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config.loader import load_config, get_mc_trials, get_random_seed
from src.simulation.scoreline_matrix import build_scoreline_matrix, extract_1x2_probabilities
from src.simulation.monte_carlo import simulate_fixture_trials, run_monte_carlo_dataset

config = load_config()
num_trials = get_mc_trials(config)
random_seed = get_random_seed(config)

processed_dir = repo_root / "data" / "processed"
input_lambdas_file = processed_dir / "adjusted_lambdas.parquet"
input_matrix_file = processed_dir / "scoreline_probabilities.parquet"
output_file = processed_dir / "mc_probabilities.parquet"

print(f"Loaded config. Configured trials: {num_trials:,}, random seed: {random_seed}")

## 2. Load Input Datasets

In [ ]:
assert input_lambdas_file.exists(), f"Missing input file: {input_lambdas_file}"
df_lambdas = pd.read_parquet(input_lambdas_file)
assert len(df_lambdas) > 0, "Input adjusted_lambdas.parquet is empty!"

df_matrix_outcomes = pd.read_parquet(input_matrix_file) if input_matrix_file.exists() else None

print(f"Loaded {len(df_lambdas)} fixtures from {input_lambdas_file.name}")
df_lambdas.head(3)

## 3. Monte Carlo Statistical Convergence Analysis

In [ ]:
# NBVAL_IGNORE_OUTPUT
# Evaluate rate of convergence against exact analytical scoreline matrix baseline
sample_row = df_lambdas.dropna(subset=["adjusted_home_lambda", "adjusted_away_lambda"]).iloc[0]
h_team, a_team = sample_row["home_team"], sample_row["away_team"]
h_lam, a_lam = float(sample_row["adjusted_home_lambda"]), float(sample_row["adjusted_away_lambda"])

# Analytical truth
matrix = build_scoreline_matrix(h_lam, a_lam, max_goals=6)
true_home_win = extract_1x2_probabilities(matrix)["prob_home_win"]

trial_counts = [10, 50, 100, 250, 500, 1000, 2500, 5000, 10000, 25000, 50000]
errors = []
sim_probs = []

for N in trial_counts:
    sim_res = simulate_fixture_trials(h_lam, a_lam, num_trials=N, random_seed=random_seed)
    sim_p = sim_res["mc_prob_home_win"]
    sim_probs.append(sim_p)
    errors.append(abs(sim_p - true_home_win))

print(f"Sample Fixture: {h_team} ({h_lam:.2f}) vs {a_team} ({a_lam:.2f})")
print(f"Analytical Home Win Prob: {true_home_win:.4f}")
print(f"Monte Carlo Home Win Prob (N={num_trials:,}): {sim_probs[8]:.4f} (Error: {errors[8]:.4f})")

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.loglog(trial_counts, errors, 'o-', color='navy', label='Absolute Error |P_MC - P_Analytical|')
ax1.axhline(0.01, color='red', linestyle='--', label='1% Error Margin Threshold')
ax1.set_xlabel('Number of Monte Carlo Trials (N)')
ax1.set_ylabel('Absolute Probability Error')
ax1.set_title(f'Monte Carlo Convergence ({h_team} vs {a_team})')
ax1.grid(True, which="both", ls="--", alpha=0.5)
ax1.legend()
plt.tight_layout()
plt.show()

## 4. Run Batch Monte Carlo Simulation Across All Fixtures

In [ ]:
df_mc_outcomes = run_monte_carlo_dataset(
    df_lambdas,
    num_trials=num_trials,
    random_seed=random_seed,
)

valid_mc = df_mc_outcomes.dropna(subset=["mc_prob_home_win"])
print(f"Computed Monte Carlo outcome probabilities ({num_trials:,} trials) for {len(valid_mc)} / {len(df_mc_outcomes)} fixtures.")

if df_matrix_outcomes is not None:
    joined = valid_mc.merge(df_matrix_outcomes[["fixture_id", "prob_home_win", "prob_draw", "prob_away_win"]], on="fixture_id")
    diff_home = np.abs(joined["mc_prob_home_win"] - joined["prob_home_win"]).mean()
    diff_draw = np.abs(joined["mc_prob_draw"] - joined["prob_draw"]).mean()
    diff_away = np.abs(joined["mc_prob_away_win"] - joined["prob_away_win"]).mean()
    print(f"Mean Absolute Difference vs Analytical Matrix -> Home Win: {diff_home:.4f}, Draw: {diff_draw:.4f}, Away Win: {diff_away:.4f}")

valid_mc[["fixture_id", "home_team", "away_team", "mc_prob_home_win", "mc_prob_draw", "mc_prob_away_win", "mc_prob_over_2_5", "mc_prob_btts"]].head(5)

## 5. Save Output Artifact

In [ ]:
# NBVAL_IGNORE_OUTPUT
output_file.parent.mkdir(parents=True, exist_ok=True)
df_mc_outcomes.to_parquet(output_file, index=False)
print(f"Saved {len(df_mc_outcomes)} rows with Monte Carlo outcome probabilities to: {output_file}")